In [14]:
# pip install langchain_huggingface

In [15]:
# pip install langchain_community

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import spacy
import re
import contractions
from textblob import TextBlob

# 1. Load the document

In [17]:
data=open('data.txt',encoding='utf-8').read()

# 2. Text Normalization

### Converting all characters into lower

In [18]:
data=data.lower()

### Removing extra space

In [19]:
data=re.sub(r'\{2,}','',data)

### Removing numbers like 1,2 3,4.....

In [20]:
# data=re.sub(r'\d+\.','',data)
# data

### Contractions

In [21]:
data=contractions.fix(data)

### Removing Special characters and punctuations

In [22]:
data=re.sub(r'[^0-9a-zA-Z\s]','',data)
data

'artificial intelligence ai is one of the fastestgrowing fields in technology it enables machines to simulate human intelligence and solve complex problems ai includes several domains such as machine learning deep learning natural language processing and computer vision\n\nmachine learning allows computers to learn from historical data and make predictions it can be divided into supervised learning unsupervised learning and reinforcement learning supervised learning uses labeled data while unsupervised learning finds hidden patterns in unlabeled data\n\ndeep learning is based on artificial neural networks it has revolutionized image recognition speech processing and language translation popular frameworks for deep learning include tensorflow and pytorch\n\nnatural language processing nlp enables computers to understand and generate human language nlp applications include chatbots sentiment analysis machine translation and text summarization tokenization stemming lemmatization and chunk

### Textblob

In [23]:
# values=TextBlob(data).correct()
# values

### Spacy and Lemmatization

In [24]:
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)

updated_tokens=[token.lemma_ for token in tokens if not token.is_stop]
data=' '.join(updated_tokens).strip()
data

'artificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing computer vision \n\n machine learning allow computer learn historical datum prediction divide supervise learning unsupervised learning reinforcement learning supervise learning use label datum unsupervised learning find hidden pattern unlabeled datum \n\n deep learning base artificial neural network revolutionize image recognition speech processing language translation popular framework deep learning include tensorflow pytorch \n\n natural language processing nlp enable computer understand generate human language nlp application include chatbot sentiment analysis machine translation text summarization tokenization stem lemmatization chunk common nlp preprocessing technique \n\n computer vision focus enable machine interpret visual information facial recognition autonomous vehicle medical imaging

In [25]:
# tokens.ents

### Chunking(converted doc -> chunks)

In [26]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)
chunks = splitter.create_documents([data])
chunks

[Document(metadata={}, page_content='artificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing'),
 Document(metadata={}, page_content='deep learn natural language processing computer vision'),
 Document(metadata={}, page_content='machine learning allow computer learn historical datum prediction divide supervise learning unsupervised learning reinforcement learning supervise learning use label datum unsupervised learning find'),
 Document(metadata={}, page_content='label datum unsupervised learning find hidden pattern unlabeled datum'),
 Document(metadata={}, page_content='deep learning base artificial neural network revolutionize image recognition speech processing language translation popular framework deep learning include tensorflow pytorch'),
 Document(metadata={}, page_content='natural language processing nlp enable computer understand generate hum

In [27]:
print(chunks[0].page_content)

artificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing


In [28]:
chunks[0].metadata={'filename':{'data.txt'}}
chunks

[Document(metadata={'filename': {'data.txt'}}, page_content='artificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing'),
 Document(metadata={}, page_content='deep learn natural language processing computer vision'),
 Document(metadata={}, page_content='machine learning allow computer learn historical datum prediction divide supervise learning unsupervised learning reinforcement learning supervise learning use label datum unsupervised learning find'),
 Document(metadata={}, page_content='label datum unsupervised learning find hidden pattern unlabeled datum'),
 Document(metadata={}, page_content='deep learning base artificial neural network revolutionize image recognition speech processing language translation popular framework deep learning include tensorflow pytorch'),
 Document(metadata={}, page_content='natural language processing nlp enable computer

In [29]:
print(type(chunks[0]))

<class 'langchain_core.documents.base.Document'>


In [30]:
print(len(chunks))

234


### Chunk Embeddings(converting chunks to vectors)

In [31]:
embedding_model=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-miniLM-L6-V2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2988.08it/s]


In [32]:
vectordb=FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
vectordb

In [33]:
user_query='What is Machine Learning ?'
r_chunks=vectordb.similarity_search(user_query)

In [34]:
updated_r_chunks=set()

for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)
updated_r_chunks
R_text='\n'.join(updated_r_chunks)
R_text

'machine learning allow computer learn historical datum prediction divide supervise learning unsupervised learning reinforcement learning supervise learning use label datum unsupervised learning find\nartificial intelligence ai fastestgrowe field technology enable machine simulate human intelligence solve complex problem ai include domain machine learn deep learn natural language processing\ncomputer vision focus enable machine interpret visual information facial recognition autonomous vehicle medical imaging surveillance system image classification object detection image segmentation\ngenerative ai advanced area ai create new content text image music video large language model llm like gpt example generative ai system model train massive dataset answer question write code'

In [35]:
def rag_query(query,k=2):
    query_embeddings=embedding_model.encode(query).astype('float32')
    query_embeddings=query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index=index_faiss_db.search(query_embeddings,k=k)
    R_chunks=[chunks[i] for i in index[0]]
    R_str=' '.join(R_chunks)
    return R_str
    prompt=f'''
        You're an helpful assistant
        Assigned Task for you : Structure my output => {R_str}
        Note : 
        1) Don't add extra contents just structure mentioned output 
        2) If there is mistake in output correct or else keep the original output with structured result.        
    '''
    import os
    import requests

    API_URL = "https://router.huggingface.co/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
    }

    def query(payload):
        response = requests.post(API_URL, headers=headers, json=payload)
        return response.json()

    response = query({
        "messages": [
            {
                "role": "user",
                "content": "prompt"
            }
        ],
        "model": "deepseek-ai/DeepSeek-R1:novita"
    })
    return response
user_prompt='Explain Machine Learning?'
user_prompt=re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

response=rag_query(user_prompt,k=2)
print(response)

AttributeError: 'HuggingFaceEmbeddings' object has no attribute 'encode'